# 05.4 — Seq2Seq with Encoder-Decoder

**Problem it solves:** Map a variable-length input sequence to a variable-length output sequence. Machine translation, summarization, dialogue.

**Architecture:** Encoder LSTM reads input → compresses to context vector → Decoder LSTM generates output one token at a time conditioned on context.

**The information bottleneck problem:** The entire input is compressed into one fixed-size vector. Long inputs lose information. This is what attention (next module) fixes.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Simple transliteration task: digits -> words
# 'one two three' -> '1 2 3'

WORD2NUM = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
}

SRC_VOCAB = ['<PAD>', '<SOS>', '<EOS>'] + list('abcdefghijklmnopqrstuvwxyz ')
TGT_VOCAB = ['<PAD>', '<SOS>', '<EOS>'] + list('0123456789 ')

src2idx = {c: i for i, c in enumerate(SRC_VOCAB)}
tgt2idx = {c: i for i, c in enumerate(TGT_VOCAB)}
idx2tgt = {i: c for c, i in tgt2idx.items()}

def make_pairs(n=1000):
    pairs = []
    words = list(WORD2NUM.keys())
    for _ in range(n):
        length = random.randint(1, 4)
        w = [random.choice(words) for _ in range(length)]
        src = ' '.join(w)
        tgt = ' '.join(WORD2NUM[x] for x in w)
        pairs.append((src, tgt))
    return pairs

def encode(text, vocab_map, max_len=40):
    ids = [vocab_map.get(c, 0) for c in text]
    return ids[:max_len]

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, n_layers, batch_first=True)
    
    def forward(self, x):
        # x: [batch, seq_len]
        embedded = self.embedding(x)            # [batch, seq_len, embed]
        outputs, (h, c) = self.lstm(embedded)
        # h, c: [n_layers, batch, hidden]
        return outputs, h, c

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h, c):
        # x: [batch, 1] (single token)
        embedded = self.embedding(x)            # [batch, 1, embed]
        output, (h, c) = self.lstm(embedded, (h, c))
        pred = self.fc(output.squeeze(1))       # [batch, vocab_size]
        return pred, h, c

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_sos, tgt_eos, tgt_vocab_size):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_sos = tgt_sos
        self.tgt_eos = tgt_eos
        self.tgt_vocab_size = tgt_vocab_size
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        outputs = torch.zeros(batch_size, tgt_len, self.tgt_vocab_size)
        
        # Encode
        _, h, c = self.encoder(src)
        
        # Decode: start with <SOS> token
        dec_input = tgt[:, 0:1]  # <SOS>
        
        for t in range(1, tgt_len):
            pred, h, c = self.decoder(dec_input, h, c)
            outputs[:, t] = pred
            
            # Teacher forcing: use ground truth or model's prediction
            use_teacher_forcing = random.random() < teacher_forcing_ratio
            dec_input = tgt[:, t:t+1] if use_teacher_forcing else pred.argmax(1).unsqueeze(1)
        
        return outputs
    
    def translate(self, src_text: str, max_len=20) -> str:
        """Greedy decoding."""
        self.eval()
        with torch.no_grad():
            src_ids = torch.tensor([encode(src_text, src2idx)], dtype=torch.long)
            _, h, c = self.encoder(src_ids)
            
            dec_input = torch.tensor([[self.tgt_sos]])
            result = []
            
            for _ in range(max_len):
                pred, h, c = self.decoder(dec_input, h, c)
                next_token = pred.argmax(1).item()
                if next_token == self.tgt_eos:
                    break
                result.append(idx2tgt.get(next_token, '?'))
                dec_input = torch.tensor([[next_token]])
        
        return ''.join(result)

# Build and train
EMBED_DIM = 32
HIDDEN = 64

encoder = Encoder(len(SRC_VOCAB), EMBED_DIM, HIDDEN)
decoder = Decoder(len(TGT_VOCAB), EMBED_DIM, HIDDEN)
model = Seq2Seq(encoder, decoder, tgt2idx['<SOS>'], tgt2idx['<EOS>'], len(TGT_VOCAB))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

pairs = make_pairs(500)

def make_batch(pairs, max_src=40, max_tgt=20):
    srcs, tgts = [], []
    for src, tgt in pairs:
        s = [src2idx.get('<SOS>',1)] + encode(src, src2idx, max_src)
        t = [tgt2idx.get('<SOS>',1)] + encode(tgt, tgt2idx, max_tgt) + [tgt2idx.get('<EOS>',2)]
        srcs.append(s[:max_src]); tgts.append(t[:max_tgt])
    # Pad
    max_s = max(len(s) for s in srcs)
    max_t = max(len(t) for t in tgts)
    srcs = [s + [0]*(max_s-len(s)) for s in srcs]
    tgts = [t + [0]*(max_t-len(t)) for t in tgts]
    return torch.tensor(srcs), torch.tensor(tgts)

for epoch in range(20):
    model.train()
    total_loss = 0
    random.shuffle(pairs)
    for i in range(0, len(pairs), 32):
        batch = pairs[i:i+32]
        src, tgt = make_batch(batch)
        optimizer.zero_grad()
        output = model(src, tgt)  # [batch, tgt_len, vocab]
        loss = criterion(output[:, 1:].reshape(-1, len(TGT_VOCAB)), tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d}: loss={total_loss/len(pairs)*32:.4f}")

In [ ]:
# Test the model
test_inputs = ['one', 'two three', 'one two three four', 'five six seven eight nine']
print("Translation results:")
for text in test_inputs:
    prediction = model.translate(text)
    expected = ' '.join(WORD2NUM[w] for w in text.split())
    correct = '✓' if prediction.strip() == expected else '✗'
    print(f"  {text:30} -> {prediction!r:15} (expected {expected!r}) {correct}")

print()
print("The information bottleneck: the encoder must compress 'one two three four five'")
print("into a single vector. Long sequences lose information.")
print("This is why attention (next module) was invented.")

## Summary

**Seq2Seq architecture:**
- Encoder: reads input, produces final hidden state
- Decoder: conditioned on encoder state, generates output autoregressively
- Teacher forcing: use gold labels during training (faster), model's own predictions at test time

**The bottleneck problem:**
- All information must pass through a fixed-size vector
- 'five eight zero seven two nine' → one 64-dim vector → decoder must recover all 6 digits
- Fails on long sequences

**Attention solves this:** Instead of one context vector, decoder can access ALL encoder hidden states, weighted by relevance at each decoding step.

---
**Next:** 05.5 — Beam Search